# WtDataStorage 框架
```mermaid
graph TD
    subgraph "接口定义层 (Interfaces)"
        direction LR
        A_IDataWriter["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>IDataWriter.h</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 定义数据写入器的核心接口(IDataWriter), 负责将实时行情(Tick, K线, 订单流)写入存储系统。<br/><b>意义:</b> 抽象了数据持久化的统一规范, 使得上层模块无需关心具体的存储实现。<br/><b>用法:</b> 由WtDataWriter类实现此接口, DataManager通过此接口调用写入功能。</div>"]
        D_IRdmDtReader["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>IRdmDtReader.h</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 定义随机数据读取器的接口, 提供按时间范围, 按数量等多种方式随机访问历史和实时数据的功能。<br/><b>意义:</b> 为回测引擎和数据分析工具提供了灵活, 高效的数据查询规范。<br/><b>用法:</b> 由WtRdmDtReader类实现, 供策略回测或数据分析时调用。</div>"]
        E_IBtDtReader["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>IBtDtReader.h</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 定义回测数据读取器的接口, 专门用于在回测过程中按顺序加载原始历史数据文件。<br/><b>意义:</b> 抽象了回测时从物理存储加载数据的过程, 支持不同的数据源(如本地文件, 数据库)。<br/><b>用法:</b> 由WtBtDtReader类实现, 在回测开始前, 由框架调用以加载所需数据。</div>"]
        IDataReader_H["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>IDataReader.h</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 定义了更高层次的数据读取器(IDataReader)和事件回调(IDataReaderSink)接口, 负责管理数据缓存, 响应时间推进, 并生成K线闭合等事件。<br/><b>意义:</b> 是策略回测中事件驱动机制的核心, 将原始的数据读取行为转换为策略可消费的`on_bar`等事件。<br/><b>用法:</b> 由`WtDataReader`实现, 在回测引擎中作为“行情源”使用。</div>"]
    end

    subgraph "核心实现层 (Implementations)"
        direction TB
        F_WtDataWriter["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>WtDataWriter.h/.cpp</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 实现了IDataWriter接口, 是数据写入功能的核心。负责将各类实时行情数据通过内存映射文件高效写入磁盘, 并在盘后进行历史数据转储。<br/><b>区别:</b> 它是数据“写”操作的具体执行者。<br/><b>用法:</b> 在WtDtCore模块中被DataManager动态加载和调用, 是数据持久化的主力。</div>"]
        G_WtRdmDtReader["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>WtRdmDtReader.h/.cpp</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 实现了IRdmDtReader接口, 提供强大的随机数据访问能力。它能够智能地组合历史数据文件和实时缓存, 提供一个统一的数据视图。<br/><b>区别:</b> 它是数据“随机读”操作的具体执行者, 核心在于高效的数据索引和切片。<br/><b>用法:</b> 在回测或数据分析时, 由上层模块直接调用, 按需获取指定范围的数据。</div>"]
        H_WtBtDtReader["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>WtBtDtReader.h/.cpp</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 实现了IBtDtReader接口, 负责在回测开始前, 从磁盘加载指定合约和周期的全部原始数据到内存缓冲区。<br/><b>区别:</b> 它是数据“批量顺序读”的执行者, 主要服务于回测开始前的数据准备阶段。<br/><b>用法:</b> 在回测引擎初始化时被调用, 为后续的回测过程提供全部所需的数据。</div>"]
        I_WtDataReader["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>WtDataReader.h/.cpp</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 实现了IDataReader接口, 是一个更上层的数据读取协调器(尽管在WtDataStorage模块中)。它管理着历史数据缓存和实时数据块, 响应策略的回调, 并在分钟线结束时触发K线闭合事件。<br/><b>区别:</b> 它更侧重于数据的“组合与事件生成”, 而非原始的读写。<br/><b>用法:</b> 在回测框架中, 它作为数据源, 随着时间的推进向策略推送数据。</div>"]
    end

    J_DataDefine["<div style='font-weight:bold; font-size: 1.2em; margin-bottom: 5px;'>DataDefine.h</div><div style='text-align:left; font-size: 0.9em;'><b>作用:</b> 定义了WonderTrader数据存储的物理格式规范。包括各种数据块(Block)的头部结构, 版本信息, 数据类型枚举等。<br/><b>意义:</b> 这是整个数据存储模块的基石, 确保了数据在磁盘上存储格式的统一性和可解析性。<br/><b>用法:</b> 被所有涉及直接读写数据文件的类(#F, #G, #H, #I)包含和使用。</div>"]

    %% 关系定义
    F_WtDataWriter -- "实现" --> A_IDataWriter
    G_WtRdmDtReader -- "实现" --> D_IRdmDtReader
    H_WtBtDtReader -- "实现" --> E_IBtDtReader
    I_WtDataReader -- "实现" --> IDataReader_H

    A_IDataWriter -.-> J_DataDefine
    D_IRdmDtReader -.-> J_DataDefine
    E_IBtDtReader -.-> J_DataDefine
    I_WtDataReader -.-> J_DataDefine

    F_WtDataWriter -- "包含并使用" --> J_DataDefine
    G_WtRdmDtReader -- "包含并使用" --> J_DataDefine
    H_WtBtDtReader -- "包含并使用" --> J_DataDefine
    I_WtDataReader -- "包含并使用" --> J_DataDefine

    %% 样式定义
    classDef interface fill:#e6f3ff,stroke:#333,stroke-width:2px,stroke-dasharray: 5 5
    classDef implementation fill:#d4edda,stroke:#333,stroke-width:2px
    classDef lowlevel fill:#f8d7da,stroke:#333,stroke-width:2px
    class A_IDataWriter,B_IHisDataDumper,D_IRdmDtReader,E_IBtDtReader,IDataReader interface
    class F_WtDataWriter,G_WtRdmDtReader,H_WtBtDtReader,I_WtDataReader implementation
    class J_DataDefine,K_Build lowlevel
```

# 存储格式定义 DataDefine.h

- **核心常量与版本控制**
  -   `const char BLK_FLAG[] = "&^%$#@!\0"`
      - 作用：数据文件的“指纹”或魔数，用于快速校验文件是否为有效的数据块。
  -   版本号宏 (`BLOCK_VERSION_*`)
      - 作用：标记数据文件的版本和格式，使得系统可以向后兼容，正确读写不同时期的数据文件。
      - 区别:
        - `_RAW` vs `_CMP`：代表数据是原始存储还是经过了压缩。
        - `_V2`：代表V2版本，其头部结构包含了数据大小信息，主要用于优化压缩数据的读取。

- **数据块类型枚举 (`BlockType`)**
  - 作用：唯一标识了数据块中存储的是哪一种数据，是数据解析的关键依据。
  - 区别 (实时 vs 历史)：
    - **实时数据块** (`BT_RT_*`)
      - 定义：用于存储**当天**的、正在产生的实时行情数据。
      - 包含：`BT_RT_Minute1`, `BT_RT_Minute5`, `BT_RT_Ticks`, `BT_RT_Cache`, `BT_RT_Trnsctn`, `BT_RT_OrdDetail`, `BT_RT_OrdQueue`。
    - **历史数据块** (`BT_HIS_*`)
      - 定义：用于存储**盘后归档**的历史数据，通常经过压缩。
      - 包含：`BT_HIS_Minute1`, `BT_HIS_Minute5`, `BT_HIS_Day`, `BT_HIS_Ticks`, `BT_HIS_Trnsctn`, `BT_HIS_OrdDetail`, `BT_HIS_OrdQueue`。

- **数据块头部结构**
  - 作用：作为所有数据块的通用元信息，提供了识别和解析数据的基础。
    - `struct _BlockHeader` (V1 基础头部)
      - 定义：包含魔数标识 `_blk_flag`、数据类型 `_type` 和版本号 `_version`。
      - 意义：V1 版本的标准文件头，定义了数据块的基本身份信息。
    - `struct _BlockHeaderV2` (V2 扩展头部)
      - 定义：在 V1 基础上增加了 `uint64_t _size` 字段，用于记录压缩后的数据大小。
      - 区别：`_size` 字段的加入使得解析压缩文件时无需读取整个文件就能知道有效数据的大小，提高了效率。
    - `struct _RTBlockHeader` (实时数据块头部)
      - 定义：继承自 `_BlockHeader`，并增加了 `_size` (当前数据条数) 和 `_capacity` (总容量) 字段。
      - 区别与意义：专门为动态增长的实时文件设计。文件会预分配 `_capacity` 的空间，`_size` 记录当前已写入的数据量，避免了频繁的文件扩展操作，提升写入性能。
    - `struct _RTDayBlockHeader` (每日实时数据块头部)
      - 定义：继承自 `_RTBlockHeader`，并增加了 `_date` (交易日期) 字段。
      - 区别与意义：使得实时数据块能够明确归属于某个交易日，是盘后按天转储历史数据的关键依据。

- **具体数据块实现**
    - 作用：将头部与实际的数据数组结合起来，构成一个完整的数据文件内存映像。采用C语言的“柔性数组成员”技巧(`_xxx[0]`)，将头部和数据存储在一块连续内存中，提高访问效率。
    - **实时数据块 (RT Blocks)**:
      - `_RTKlineBlock: _RTDayBlockHeader`：实时K线。
      - `_RTTickBlock: _RTDayBlockHeader`：实时Tick。
      - `_RTTransBlock: _RTDayBlockHeader`：实时逐笔成交。
      - `_RTOrdDtlBlock: _RTDayBlockHeader`：实时逐笔委托。
      - `_RTOrdQueBlock: _RTDayBlockHeader`：实时委托队列。
      - `_RTTickCache: _RTBlockHeader` & `_TickCacheItem`：特殊的全局Tick缓存，用于快速索引各合约的**最新一笔Tick**。
    - **历史数据块 (His Blocks)**:
      - **区别 (V1 vs V2)**：V1 版本直接包含数据数组，而 V2 版本则包含一个 `char _data[0]` 成员，用于存放**压缩后**的二进制数据流。
      - `_HisKlineBlock: _BlockHeader` / `_HisKlineBlockV2: _BlockHeaderV2`：历史K线。
      - `_HisTickBlock: _BlockHeader` / `_HisTickBlockV2: _BlockHeaderV2`：历史Tick。
      - `_HisTransBlock: _BlockHeader` / `_HisTransBlockV2: _BlockHeaderV2`：历史逐笔成交。
      - `_HisOrdDtlBlock: _BlockHeader` / `_HisOrdDtlBlockV2: _BlockHeaderV2`：历史逐笔委托。
      - `_HisOrdQueBlock: _BlockHeader` / `_HisOrdQueBlockV2: _BlockHeaderV2`：历史委托队列。
      - `_HisKlineBlockOld: _BlockHeader`：一个特殊的历史K线块，用于兼容使用了旧版 `WTSBarStructOld` 结构的数据格式。

# 数据读取器 WtDataReader.h/cpp
```cpp
class WtDataReader : public IDataReader
```
参考 [Includes/note.ipynb/数据管理接口层/数据读取 IDataReader.h/数据读取接口类 IDataReader](../Includes/note.ipynb)

## 成员
- **目录路径**
  - `std::string _rt_dir`：实时数据存储目录
  - `std::string _his_dir`：历史数据存储目录
- **核心管理器指针**
  - `IBaseDataMgr* _base_data_mgr`：基础数据管理器指针
  - `IHotMgr* _hot_mgr`：主力合约管理器指针
- **配置与状态**
  - `uint32_t _adjust_flag`：复权标记（位运算表示），1|2|4,1表示成交量复权，2表示成交额复权，4表示总持复权，其他待定
  - `uint64_t	_last_time`：最后时间戳
- **实时数据块缓存**
  - `RTKBlockFilesMap _rt_min1_map`：实时1分钟K线数据块映射表
  - `RTKBlockFilesMap _rt_min5_map`：实时5分钟K线数据块映射表
    - typedef wt_hashmap\<std::string, `RTKlineBlockPair`\>	RTKBlockFilesMap; 包含
      - `RTKlineBlock`*	_block：实时K线数据块指针
      - BoostMFPtr _file：内存映射文件智能指针
      - uint64_t _last_cap：最后容量（用于容量管理）
  - `TBlockFilesMap _rt_tick_map`：实时Tick数据块映射表
    - typedef wt_hashmap\<std::string, `TickBlockPair`\>	TBlockFilesMap; 包含
      - `RTTickBlock`* _block：实时Tick数据块指针
      - BoostMFPtr _file：内存映射文件智能指针
      - uint64_t _last_cap：最后容量（用于容量管理）
  - `TransBlockFilesMap _rt_trans_map`：实时逐笔成交数据块映射表
    - typedef wt_hashmap\<std::string, `TransBlockPair`\>	TransBlockFilesMap; 包含
      - `RTTransBlock`*	_block：实时逐笔成交数据块指针
      - BoostMFPtr _file：内存映射文件智能指针
      - uint64_t _last_cap：最后容量（用于容量管理）
      - std::shared_ptr< std::ofstream>	_fstream：文件流智能指针（用于数据写入）
  - `OrdDtlBlockFilesMap	_rt_orddtl_map`：实时逐笔委托数据块映射表
    - typedef wt_hashmap\<std::string, `OrdDtlBlockPair`\>	OrdDtlBlockFilesMap; 包含
      - `RTOrdDtlBlock`*	_block：实时逐笔委托数据块指针
      - BoostMFPtr _file：内存映射文件智能指针
      - uint64_t _last_cap：最后容量（用于容量管理）
      - std::shared_ptr< std::ofstream>	_fstream：文件流智能指针（用于数据写入）
  - `OrdQueBlockFilesMap	_rt_ordque_map`：实时委托队列数据块映射表
    - typedef wt_hashmap\<std::string, `OrdQueBlockPair`\>	OrdQueBlockFilesMap; 包含
      - `RTOrdQueBlock`*	_block：实时委托队列数据块指针
      - BoostMFPtr _file：内存映射文件智能指针
      - uint64_t _last_cap：最后容量（用于容量管理）
      - std::shared_ptr< std::ofstream>	_fstream：文件流智能指针（用于数据写入）
- **历史数据块缓存**
  - `HisTickBlockMap	_his_tick_map`：历史Tick数据块映射表
    - typedef wt_hashmap\<std::string, `HisTBlockPair`\>	HisTickBlockMap; 包含
      - `HisTickBlock`* _block：历史Tick数据块指针
      - uint64_t _date：交易日期（YYYYMMDD格式）
      - std::string	_buffer：数据缓冲区
  - `HisOrdDtlBlockMap _his_orddtl_map`：历史逐笔委托数据块映射表
    - typedef wt_hashmap\<std::string, `HisTransBlockPair`\>	HisTransBlockMap; 包含
      - `HisTransBlock`* _block：历史逐笔成交数据块指针
      - uint64_t _date：交易日期（YYYYMMDD格式）
      - std::string _buffer：数据缓冲区
  - `HisOrdQueBlockMap _his_ordque_map`：历史委托队列数据块映射表
    - typedef wt_hashmap\<std::string, `HisOrdDtlBlockPair`\>	HisOrdDtlBlockMap; 包含
      - `HisOrdDtlBlock`* _block：历史逐笔委托数据块指针
      - uint64_t _date：交易日期（YYYYMMDD格式）
      - std::string _buffer：数据缓冲区
  - `HisTransBlockMap _his_trans_map`：历史逐笔成交数据块映射表
    - typedef wt_hashmap\<std::string, `HisOrdQueBlockPair`\>	HisOrdQueBlockMap; 包含
      - `HisOrdQueBlock`* _block：历史委托队列数据块指针
      - uint64_t _date：交易日期（YYYYMMDD格式）
      - std::string _buffer：数据缓冲区
- **K线与复权数据缓存**
  - `BarsCache _bars_cache`：K线数据缓存
    - typedef wt_hashmap\<std::string, `BarsList`\> BarsCache;
      - std::string	_exchg：交易所代码
      - std::string	_code：合约代码
      - `WTSKlinePeriod` _period：K线周期
      - uint32_t _rt_cursor：实时数据光标位置
      - std::string _raw_code：原始合约代码
      - std::vector<`WTSBarStruct`>	_bars：K线数据向量
      - double _factor：复权因子
  - `AdjFactorMap	_adj_factors`：复权因子映射表
    - typedef wt_hashmap\<std::string, `AdjFactorList`\>	AdjFactorMap：复权因子映射表
    - typedef std::vector\<`AdjFactor`\> AdjFactorList：复权因子列表
      - uint32_t _date：交易日期
      - double _factor：复权因子值

## 方法

### IDataReader 接口实现

#### 初始化数据读取器 init

#### 分钟结束回调 onMinuteEnd
**前复权**：维持当前数据，调整所有历史数据。
**后复权**：维持历史数据，调整当前数据（及以后）。
- `_his_***_map` 中存储历史数据（复权后的）
- `_rt_***_map` 中是新来的一批实时数据（还未复权）

**该函数在每分钟结束时被调用，用于处理实时K线数据的更新**：
- 会**遍历**K线缓存 `_bars_cache` 中的每一个分钟周期的K线数据列表的引用 barsList
  - 根据 barList 的交易所代码、合约代码、周期，获取 `_rt_min1_map`/`_rt_min5_map` 对应的K线数据块
  - **遍历**其中的每个K线数据（从 barsList 的 `_rt_cursor` 开始，且时间小于 uDate 和 uTime）
    - 如果是前复权
      - 直接使用 `_sink` 的 on_bar 函数对该K线数据回调（因为当前价格不需要调整）
    - 否则即是后复权
      - open/high/low/close 都乘上 barsList 的 `_factor` 后构建新的K线数据并存到 barList 尾部（相当于复权了）
      - 使用 `_sink` 的 on_bar 函数对该复权后的尾部K线数据回调
  - 根据实际处理的K线数据条数，更新 barsList 的 `_rt_cursor`
```cpp
/*!
 * \brief 分钟结束回调函数
 * \param uDate 交易日期
 * \param uTime 结束时间
 * \param endTDate 结束交易日期（可选，默认为0）
 */
void WtDataReader::onMinuteEnd(uint32_t uDate, uint32_t uTime, uint32_t endTDate /* = 0 */)
```

#### 读取Tick数据切片 readTickSlice
```cpp
/*!
 * \brief 读取Tick数据切片
 * \param stdCode 标准合约代码
 * \param count 数据数量
 * \param etime 结束时间（可选，默认为0）
 * \return Tick数据切片指针
 */
WTSTickSlice* WtDataReader::readTickSlice(const char* stdCode, uint32_t count, uint64_t etime /* = 0 */)
```
流程：
- 使用参数结束时间 eTime 构造一个Tick数据 eTick: WTSTickStruct
- 对应于 stdCode 的节假日模板，计算参数 eTime 以及当前时刻的交易日期 endTDate 和 curTDate
- 如果是同一天
  - 从中 `_rt_tick_map` 读取对应 stdCode 的实时tick数据块 tBlock: RTTickBlock
  - 从其中取出对应 eTick 时间之前的 count 个tick数据构成 WTSTickSlice* 返回
- 否则
  - 从 `_his_tick_map` 中读取对应 stdCode 和 endTDate 的历史tick数据块
  - 如果没有
    - 构建 {_his_dir}ticks/{cInfo._exchg}/{endTDate}/{curCode}.dsb 并将其中内容读取到 `_his_tick_map[{stdCode}-{endTDate}]`
  - 从其中取出对应 eTick 时间之前的 count 个tick数据构成 WTSTickSlice* 返回

#### 读取逐笔委托数据切片 readOrdDtlSlice

#### 读取委托队列数据切片 readOrdQueSlice

#### 读取逐笔成交数据切片 readTransSlice

#### 读取K线数据切片 readKlineSlice

#### 获取指定日期的复权因子 getAdjFactorByDate

### 实时数据块获取

#### 获取实时K线数据块 getRTKilneBlock
- 根据 period 获取 `_rt_min1_map`/`_rt_min5_map` 中对应 {exchg}.{code} 键的数据引用 `block: RTKlineBlockPair`
- 考虑实时文件路径 {`_rt_dir`}{min1或min5}/{exchg}/{code}.dmb
- `block._file` 和 `block._block` 应当已经映射到这个实时文件，如果没有则应该映射
- 最后返回 `block` 的引用
```cpp
/*!
 * \brief 获取实时K线数据块
 * \param exchg 交易所代码
 * \param code 合约代码
 * \param period K线周期
 * \return 实时K线数据块对指针
 */
WtDataReader::RTKlineBlockPair* WtDataReader::getRTKilneBlock(const char* exchg, const char* code, WTSKlinePeriod period)
```

#### 获取实时Tick数据块 getRTTickBlock

#### 获取实时委托队列数据块 getRTOrdQueBlock

#### 获取实时逐笔委托数据块 getRTOrdDtlBlock

#### 获取实时逐笔成交数据块 getRTTransBlock

### 数据缓存

#### 从加载器中加载K线数据到缓存 cacheFinalBarsFromLoader
- 获取K线缓存 `_bars_cache[key]` 的引用，并依据参数设置其 _code、_period、_exchg
- 加载器调用回调函数 `_loader->loadFinalHisBars` 将加载器准备的K线数据复制到 _bars
```cpp
/*!
 * \brief 从外部加载器缓存最终K线数据
 * \param codeInfo 合约代码信息指针
 * \param key 缓存键值
 * \param stdCode 标准合约代码
 * \param period K线周期
 * \return 是否缓存成功
 */
bool WtDataReader::cacheFinalBarsFromLoader(void* codeInfo, const std::string& key, const char* stdCode, WTSKlinePeriod period)
```

#### 加载经分月合约调整的K线数据到缓存 cacheIntegratedBars
```cpp
/*!
 * \brief 加载经分月合约调整的K线数据到缓存
 * \param codeInfo 合约代码信息指针
 * \param key 缓存键值
 * \param stdCode 标准合约代码
 * \param period K线周期
 * \return 是否缓存成功
 */
bool WtDataReader::cacheIntegratedBars(void* codeInfo, const std::string& key, const char* stdCode, WTSKlinePeriod period)
```
具体流程：
- **预加载数据**：从 `{_his_dir}{pname: min1/min5/day}/{cInfo->_exchg}/{cInfo->_exchg}.{cInfo->_product}_{cInfo->_ruletag: HOT/2ND}{cInfo->_exright: -/+}.dsb` 加载K线数据到 hotAy: std::vector\<WTSBarStruct\>\*，记录其最后时间到 lastHotTime
- **读取分月合约历史** 到 secs: std::vector\<HotSection\>
  - std::string	_code：分月合约代码
  - uint32_t _s_date：开始日期，格式为YYYYMMDD
  - uint32_t _e_date：结束日期，格式为YYYYMMDD
  - double _factor：绝对价格基准比例，该分月合约的价格与历史上某个固定基准点的价格之间的比率
- 构建 barsSections: std::vector\<std::vector\<WTSBarStruct\>\*\>
- **倒序（时间从后往前）遍历每个分月合约**
  - 如果该分月合约的开始时间 < lastHotTime：不需要遍历后面的所有分月合约；如果结束时间 <= lastHotTime：被预加载数据完全包含，跳出遍历
  - **从加载器 `_loader`（函数 loadRawHisBars）获得该分月合约的K线数据，否则从 `{_his_dir}{pname: min1/min5/day}/{cInfo->_exchg}/{当前分月合约代码}.dsb`中获得**
    - 从中找到对应该分月合约起始时间和结束时间的位置 sIdx 和 eIdx
    - 计算factor：前复权 —— $\frac{当前\_factor}{最后一个分月切换的\_factor}$；后复权 —— $当前\_factor$
    - 将这些位置的open/high/low/close 乘上该 factor
    - 根据 `_adjust_flag`（调整成交量/成交额/持仓量），将 vol/money/hold 和add 除以 factor
  - 将上述调整过的部分K线数据构成一个 std::vector\<WTSBarStruct\>\* 并添加到 barsSections 尾部
- 将 hotAy 添加到 barsSections 尾部
- 从后往前将 barsSections 中的每一项复制到 `_bars_cache[key]`

#### 加载经复权因子调整的K线数据到缓存 cacheAdjustedStkBars
```cpp
/*!
 * \brief 加载经复权因子调整的K线数据到缓存
 * \param codeInfo 合约代码信息指针
 * \param key 缓存键值
 * \param stdCode 标准合约代码
 * \param period K线周期
 * \return 是否缓存成功
 */
bool WtDataReader::cacheAdjustedStkBars(void* codeInfo, const std::string& key, const char* stdCode, WTSKlinePeriod period)
```
具体流程：
- **预加载复权数据**：从 `{_his_dir}{pname: min1/min5/day}/{cInfo->_exchg}/{cInfo->_code}{cInfo->_exright: -/+}.dsb` 加载K线数据到 ayAdjusted: std::vector\<WTSBarStruct\>\*，记录其最后时间到 lastQTime
- 构建 barsSections: std::vector\<std::vector\<WTSBarStruct\>\*\>
- **从加载器 `_loader`（函数 loadRawHisBars）或 `{_his_dir}{pname: min1/min5/day}/{cInfo->_exchg}/{cInfo->_code}.dsb`中获得原始K线数据**
  - 找出对应 lastQTime 时间之后的数据到 ayRaw: std::vector\<WTSBarStruct\>\*
  - **从 `_adj_factors` 中获取对应的复权因子列表，并从后往前遍历**
    - 计算factor：前复权 —— $\frac{当前\_factor}{最后一个复权因子的\_factor}$；后复权 —— $当前\_factor$
    - **找到该复权因子生效的起始K线，遍历对应的所有K线**
      - 将这些位置的open/high/low/close 乘上该 factor
      - 根据 `_adjust_flag`（调整成交量/成交额/持仓量），将 vol/money/hold 和add 除以 factor
- 将 ayAdjusted 添加到 barsSections 尾部
- 从后往前将 barsSections 中的每一项复制到 `_bars_cache[key]`

#### 加载各类型的K线数据到缓存 cacheHisBarsFromFile
```cpp
/*!
 * \brief 从文件缓存历史K线数据
 * \param codeInfo 合约代码信息指针
 * \param key 缓存键值
 * \param stdCode 标准合约代码
 * \param period K线周期
 * \return 是否缓存成功
 */
bool WtDataReader::cacheHisBarsFromFile(void* codeInfo, const std::string& key, const char* stdCode, WTSKlinePeriod period)
```
具体流程：
- 如果是 *期货连续合约*（有 `HOT/2ND` 标识）
  - 使用 cacheIntegratedBars 加载经分月合约调整的K线数据到缓存
- 否则如果是 *股票复权合约*（有 `-/+` 标识）
  - 使用 cacheAdjustedStkBars 加载经复权因子调整的K线数据到缓存
- 否则（即是简单合约）
  - **从加载器 `_loader`（函数 loadRawHisBars）或 `{_his_dir}{pname: min1/min5/day}/{cInfo->_exchg}/{cInfo->_code}.dsb`中获得K线数据**
  - 然后复制到`_bars_cache[key]`

### 复权因子处理

#### 从文件加载股票复权因子 loadStkAdjFactorsFromFile

#### 从加载器加载股票复权因子 loadStkAdjFactorsFromLoader

#### 获取复权因子列表 getAdjFactors

# 数据写入器 WtDataWriter.h/cpp
```cpp
class WtDataWriter : public IDataWriter
```
参考 [Includes/note.ipynb/数据管理接口层/数据写入 IDataWriter.h/数据写入接口类 IDataWriter](../Includes/note.ipynb)

## 成员
-   **核心管理器指针**
    -   `IBaseDataMgr* _bd_mgr`：基础数据管理器指针，用于获取合约、交易日等静态基础信息。

-   **配置与状态标志**
    -   `std::string _base_dir`：数据存储的根目录路径。
    -   `std::string _cache_file`：实时Tick缓存文件的名称。
    -   `uint32_t _log_group_size`：定义了批量日志输出的条数阈值。
    -   `bool _async_proc`：异步处理开关，`true` 表示数据将进入队列由后台线程处理，`false` 表示在当前线程同步处理。
    -   `bool _terminated`：线程终止标志，用于通知后台线程安全退出。
    -   `bool _save_tick_log`：是否将收到的Tick数据额外保存为CSV日志文件。
    -   `bool _skip_notrade_tick`：是否跳过没有成交的Tick（即价格未变化）。
    -   `bool _skip_notrade_bar`：是否跳过完全没有成交记录的K线周期。
    -   **数据写入禁用标志**
        -   `bool _disable_his`：是否禁用所有历史数据（包括K线和Tick）的转储功能。
        -   `bool _disable_tick`：是否禁用实时Tick数据的写入。
        -   `bool _disable_min1`：是否禁用实时1分钟K线的合成与写入。
        -   `bool _disable_min5`：是否禁用实时5分钟K线的合成与写入。
        -   `bool _disable_day`：是否禁用日K线的合成与写入。
        -   `bool _disable_trans`：是否禁用实时逐笔成交数据的写入。
        -   `bool _disable_ordque`：是否禁用实时委托队列数据的写入。
        -   `bool _disable_orddtl`：是否禁用实时逐笔委托数据的写入。
    -   `uint32_t _min_price_mode`：分钟线价格模式，`0`为常规模式，`1`为记录买卖价的特殊模式（主要用于期权等不活跃品种）。
    -   `std::map<std::string, uint32_t> _proc_date`：记录每个交易会话（Session）的盘后数据处理日期，用于防止重复处理。

-   **实时数据块缓存**
    -   `KBlockFilesMap _rt_min1_blocks`：实时1分钟K线数据块映射表。
        -   typedef wt_hashmap<std::string, `KBlockPair`\*> `KBlockFilesMap`; 包含:
            -   `RTKlineBlock* _block`：指向内存映射文件中的实时K线数据块。
            -   `BoostMFPtr _file`：内存映射文件智能指针。
            -   `SpinMutex _mutex`：自旋锁，用于保护该数据块的线程安全访问。
            -   `uint64_t _lasttime`：最后更新时间戳。
    -   `KBlockFilesMap _rt_min5_blocks`：实时5分钟K线数据块映射表（结构同上）。
    -   `TickBlockFilesMap _rt_ticks_blocks`：实时Tick数据块映射表。
        -   typedef wt_hashmap<std::string, `TickBlockPair`\*> `TickBlockFilesMap`; 包含:
            -   `RTTickBlock* _block`：指向内存映射文件中的实时Tick数据块。
            -   `BoostMFPtr _file`：内存映射文件智能指针。
            -   `SpinMutex _mutex`：自旋锁。
            -   `uint64_t _lasttime`：最后更新时间戳。
            -   `std::shared_ptr<std::ofstream> _fstream`：当`_save_tick_log`为`true`时，用于写入CSV日志的文件流指针。
    -   `TransBlockFilesMap _rt_trans_blocks`：实时逐笔成交数据块映射表。
        -   typedef wt_hashmap<std::string, `TransBlockPair`\*> `TransBlockFilesMap`; 包含:
            -   `RTTransBlock* _block`：指向内存映射文件中的实时逐笔成交数据块。
            -   `BoostMFPtr _file`：内存映射文件智能指针。
            -   `SpinMutex _mutex`：自旋锁。
            -   `uint64_t _lasttime`：最后更新时间戳。
    -   `OrdDtlBlockFilesMap _rt_orddtl_blocks`：实时逐笔委托数据块映射表。
        -   typedef wt_hashmap<std::string, `OrdDtlBlockPair`\*> `OrdDtlBlockFilesMap`; 包含:
            -   `RTOrdDtlBlock* _block`：指向内存映射文件中的实时逐笔委托数据块。
            -   `BoostMFPtr _file`：内存映射文件智能指针。
            -   `SpinMutex _mutex`：自旋锁。
            -   `uint64_t _lasttime`：最后更新时间戳。
    -   `OrdQueBlockFilesMap _rt_ordque_blocks`：实时委托队列数据块映射表。
        -   typedef wt_hashmap<std::string, `OrdQueBlockPair`\*> `OrdQueBlockFilesMap`; 包含:
            -   `RTOrdQueBlock* _block`：指向内存映射文件中的实时委托队列数据块。
            -   `BoostMFPtr _file`：内存映射文件智能指针。
            -   `SpinMutex _mutex`：自旋锁。
            -   `uint64_t _lasttime`：最后更新时间戳。

-   **全局Tick缓存**
    -   `RTTickCache* _tick_cache_block`：指向全局Tick缓存数据块的指针，**存储合约的最新Tick**。
        -   `SpinMutex _lck_tick_cache`：用于保护全局Tick缓存的自旋锁。
        -   `wt_hashmap<std::string, uint32_t> _tick_cache_idx`：一个索引映射表，将合约代码映射到`_tick_cache_block`中的数组下标。
        -   `BoostMFPtr _tick_cache_file`：全局Tick缓存的内存映射文件智能指针。

-   **异步处理与线程管理**
    -   `std::queue<_TaskInfo> _tasks`：任务队列，当`_async_proc`为`true`时，所有待处理的数据都被包装成`_TaskInfo`并放入此队列。
        -   `struct alignas(64) _TaskInfo`：任务信息结构体，包含:
            -   `WTSObject* _obj`：指向数据对象的指针（如`WTSTickData`），采用引用计数管理。
            -   `uint64_t _type`：任务类型（0代表Tick, 1代表委托队列等）。
            -   `uint32_t _flag`：任务标志。
    -   `StdThreadPtr _task_thrd`：任务处理线程的智能指针，负责从`_tasks`队列中取出并处理数据。
        -   `StdUniqueMutex _task_mtx`：用于保护`_tasks`队列的互斥锁。
        -   `StdCondVariable _task_cond`：用于在`_tasks`队列为空时挂起、有新任务时唤醒`_task_thrd`线程的条件变量。
    -   `StdThreadPtr _proc_thrd`：盘后处理线程的智能指针，负责执行历史数据转储任务。
        -   `std::queue<std::string> _proc_que`：盘后处理队列，存储需要进行历史数据转储的合约代码。
            -   `StdUniqueMutex _proc_mtx` 和 `StdCondVariable _proc_cond`：用于`_proc_que`队列的同步与线程通信。
    -   `StdThreadPtr _proc_chk`：一个独立的检查线程，定期清理超时的、未被访问的实时数据块缓存，释放内存映射。


## 方法

### IDataWriter 接口实现

#### 初始化数据写入器 init
- 初始化相关内容
- 将 `_tick_cache_file`、`_tick_cache_block` 与文件 `{_base_dir}{_cache_file}` 关联
- 启动检查线程 check_loop()：将实时数据映射表 `rt_*_blocks`中，最后更新时间早于当前时间 600 秒的数据块释放掉
```cpp
/*!
 * \brief 初始化数据写入器
 * \param params 配置参数
 * \param sink 数据写入回调接口
 * \return 是否初始化成功
 */
bool WtDataWriter::init(WTSVariant* params, IDataWriterSink* sink)
```

#### 写入Tick数据 writeTick
```cpp
/*!
 * \brief 写入Tick数据
 * \param curTick 当前Tick数据
 * \param procFlag 处理标志
 * \return 是否写入成功
 */
bool WtDataWriter::writeTick(WTSTickData* curTick, uint32_t procFlag)
{
	if (curTick == NULL)
		return false;

	if (_async_proc)    // 如果启用异步处理
		pushTask(TaskInfo(curTick, 0, procFlag));
	else                // 如果同步处理
		procTick(curTick, procFlag);

	return true;
}
```

#### 写入委托队列数据 writeOrderQueue
```cpp
/*!
 * \brief 写入委托队列数据
 * \param curOrdQue 当前委托队列数据
 * \return 是否写入成功
 */
bool WtDataWriter::writeOrderQueue(WTSOrdQueData* curOrdQue)
{
	if (curOrdQue == NULL || _disable_ordque)
		return false;
	if (_async_proc)    // 如果启用异步处理
		pushTask(TaskInfo(curOrdQue, 1));
	else                // 如果同步处理
		procQueue(curOrdQue);

	return true;
}
```

#### 写入委托详情数据 writeOrderDetail
```cpp
bool WtDataWriter::writeOrderDetail(WTSOrdDtlData* curOrdDtl)
{
	if (curOrdDtl == NULL || _disable_orddtl)
		return false;

	if (_async_proc)
		pushTask(TaskInfo(curOrdDtl, 2));
	else
		procOrder(curOrdDtl);

	return true;
}
```

#### 写入逐笔成交数据 writeTransaction
```cpp
/*!
 * \brief 写入逐笔成交数据
 * \param curTrans 当前逐笔成交数据
 * \return 是否写入成功
 */
bool WtDataWriter::writeTransaction(WTSTransData* curTrans)
{
	if (curTrans == NULL || _disable_orddtl)
		return false;

	if (_async_proc)    // 如果启用异步处理
		pushTask(TaskInfo(curTrans, 3));
	else                // 如果同步处理
		procTrans(curTrans);

	return true;
}
```

#### 转储历史数据 transHisData
- **如果 sid 是交易时段标识**
  - 获取对应的所有商品（交易所.品种）
    - 获取对应的所有合约
      - 将完整合约代码添加到 `_proc_que`
  - 最后将 `MARK.{sid}` 添加到 `_proc_que`
- **否则（清除缓存命令 `CMD_CLEAR_CACHE`）**
  - 将 sid 添加到 `_proc_que`
- **如果 `_proc_thrd` 为启动：将函数 proc_loop 作为线程函数启动**
```cpp
/*!
 * \brief 转储历史数据
 * \param sid 会话标识
 */
void WtDataWriter::transHisData(const char* sid)
```

### 异步与后台任务

#### 线程：推送任务到队列 pushTask
将任务推送到 `_tasks`，如果 `_task_thrd` 不存在则创建并启动循环处理线程。
```cpp
/*!
 * \brief 推送任务到队列
 * \param task 任务信息
 */
void WtDataWriter::pushTask(const TaskInfo& task)
```

#### 线程：数据处理循环（异步任务） proc_loop
当 `_proc_que` 不空时，循环处理其首部 fullcode：
- **如果是清除缓存命令 `CMD_CLEAR_CACHE`**
  - 对于 `_tick_cache_block` 中的有效合约
    - 将全局Tick缓存 `_tick_cache_block` 中的最新Tick数据写入输出流（最后创建 `his/snapshot/{当天日期}.csv` 并写入）
  - 而对于无效合约
    - 删除文件 `{_base_dir}rt/ticks/{交易所}/{合约}.dmb`
  - 如果有效合约数目和全局Tick缓存中的数目不一致（说明有的过期了）：重建 `_tick_cache_block`
  - 清理 `{_base_dir}rt/min1|min5|ticks|orders|queue|trans/"` 下的所有文件
- **否则如果是以MARK.开头命令 `MARK.{交易时段sid}`**
  - 编辑 `{_base_dir}/marker.ini` 文件：在 [markers] 段下，添加 sid: curDate 的键值对
- **获取 `_rt_ticks|trans|orddtl|ordque_blocks` 中对应当前日期的数据块**
  - 遍历转储器 `_dumpers` 中的每一个：调用 `dumpHisTicks` 对该数据块进行转储
  - 将该数据块写入 `{_base_dir}/his/ticks|trans|orders|queue|/{交易所}/{日期}/{合约}.dsb`
```cpp
void WtDataWriter::proc_loop()
```

#### 线程：数据检查循环（缓存清理） check_loop
将实时数据映射表中，最后更新时间早于当前时间 600 秒的数据块释放掉。
- `_rt_ticks_blocks`
- `_rt_trans_blocks`
- `_rt_orddtl_blocks`
- `_rt_ordque_blocks`
- `_rt_min1_blocks`
- `_rt_min5_blocks`
```cpp
void WtDataWriter::check_loop()
```

### 内部数据处理

#### 处理Tick数据 procTick
循环处理：
- 如果 `_sink` 检查当前交易时段可以接收数据（如果不满足跳出循环）
  - 成功更新全局Tick缓存 `_tick_cache_block`（如果不满足跳出循环）
    - 将Tick数据写入实时Tick数据块 `_rt_ticks_blocks`
    - 将Tick数据合成并写入实时K线数据块 `_rt_min1_blocks/_rt_min5_blocks`
    - 使用 `_sink` 的函数 `broadcastTick` 广播Tick数据
```cpp
/*!
 * \brief 处理Tick数据
 * \param curTick 当前Tick数据
 * \param procFlag 处理标志
 */
void WtDataWriter::procTick(WTSTickData* curTick, uint32_t procFlag)
```

#### 处理委托队列数据 procQueue
循环处理：
- 如果 `_sink` 检查当前交易时段可以接收数据（如果不满足跳出循环）
  - 将委托队列数据写入实时委托队列数据块 `_rt_ordque_blocks`
  - 使用 `_sink` 的函数 `broadcastOrdQue` 广播委托队列数据
```cpp
/*!
 * \brief 处理委托队列数据
 * \param curOrdQue 当前委托队列数据
 */
void WtDataWriter::procQueue(WTSOrdQueData* curOrdQue)
```

#### 处理委托详情数据 procOrder
循环处理：
- 如果 `_sink` 检查当前交易时段可以接收数据（如果不满足跳出循环）
  - 将委托数据写入实时委托数据块 `_rt_orddtl_blocks`
  - 使用 `_sink` 的函数 `broadcastOrdDtl` 广播委托队列数据
```cpp
void WtDataWriter::procOrder(WTSOrdDtlData* curOrdDtl)
```

#### 处理逐笔成交数据 procTrans
循环处理：
- 如果 `_sink` 检查当前交易时段可以接收数据（如果不满足跳出循环）
  - 将逐笔成交写入实时逐笔成交数据块 `_rt_trans_blocks`
  - 使用 `_sink` 的函数 `broadcastTrans` 广播委托队列数据
```cpp
void WtDataWriter::procTrans(WTSTransData* curTrans)
```

### 缓存与K线管理

#### 加载全局Tick缓存 loadCache
将 `_tick_cache_file`、`_tick_cache_block` 与文件 `{_base_dir}{_cache_file}` 关联。
```cpp
void WtDataWriter::loadCache()
```

#### 更新全局Tick缓存 updateCache
使用 curTick 更新实时Tick缓存 `_tick_cache_block`。
- 如果 `_tick_cache_block` 中没有 ct 对应的合约，就先创建对应的缓存项
- curTick 的交易日期必须大于等于缓存项的交易日期 `_date`，否则忽略
- 如果 porcFlag = 0（curTick本身既有增量信息又有总量信息）：直接将curTick复制到缓存中；否则要先计算增量
  - 增量成交量：curTick的volume = curTick的total_volume - 缓存项的total_volume
  - 增量成交额：curTick的turn_over = curTick的total_turnover - 缓存项的total_turnover
  - 持仓变化：curTick的diff_interest = curTick的open_interest - 缓存项的open_interest
```cpp
/*!
 * \brief 更新Tick缓存
 * \param ct 合约信息
 * \param curTick 当前Tick数据
 * \param procFlag 处理标志
 * \return 是否更新成功
 */
bool WtDataWriter::updateCache(WTSContractInfo* ct, WTSTickData* curTick, uint32_t procFlag)
```

#### 将Tick数据写入实时Tick数据块 pipeToTicks
```cpp
/*!
 * \brief 将数据传递给实时Tick数据块
 * \param ct 合约信息
 * \param curTick 当前Tick数据
 */
void WtDataWriter::pipeToTicks(WTSContractInfo* ct, WTSTickData* curTick)
```

#### 将Tick数据合成并写入实时K线数据块 pipeToKlines
```cpp
/*!
 * \brief 将Tick数据合成并写入实时K线文件_rt_min1_blocks/_rt_min5_blocks
 * \param ct 合约信息
 * \param curTick 当前Tick数据
 */
void WtDataWriter::pipeToKlines(WTSContractInfo* ct, WTSTickData* curTick)
```

### 盘后数据转储

#### 转储日线数据 dump_day_data
将日线数据 newBar 转储到文件 `his/day/{ct->getExchg()}/{ct->getCode()}.dsb`
```cpp
/*!
 * \brief 转储日线数据
 * \param ct 合约信息
 * \param newBar 新的日线K线数据
 * \return 是否转储成功
 */
bool WtDataWriter::dump_day_data(WTSContractInfo* ct, WTSBarStruct* newBar)
```

#### 获取实时K线数据并转储 dump_bars_to_file
从 `_tick_cache_block` 获取对应合约 ct 的最新 tick 数据并转储。
- `_tick_cache_block` 转储到 `his/day/{ct->getExchg()}/{ct->getCode()}.dsb`
- `_rt_min1_blocks` 转储到 `his/min1/{ct->getExchg()}/{ct->getCode()}.dsb`
- `_rt_min5_blocks` 转储到 `his/min5/{ct->getExchg()}/{ct->getCode()}.dsb`

#### 通过扩展转储器转储K线 dump_bars_via_dumper
与函数 dump_bars_to_file 的区别：遍历 `_dumpers`，调用其 dumpHisBars 方法进行转储。

### 实时数据块管理

#### 获取或创建K线数据块 getKlineBlock
根据 period 获取实时K线映射表 `_rt_min1_blocks`/`_rt_min5_blocks` 中对应 ct 的数据块的引用。
- 如果不存在该数据块，创建 `{_base_dir}rt/{min1/min5}/{ct->getExchg()}/{ct->getCode()}.dmb` 并关系起来
```cpp
/*!
 * \brief 获取K线数据块
 * \param ct 合约信息
 * \param period K线周期
 * \param bAutoCreate 是否自动创建
 * \return K线数据块指针
 */
WtDataWriter::KBlockPair* WtDataWriter::getKlineBlock(WTSContractInfo* ct, WTSKlinePeriod period, bool bAutoCreate /* = true */)
```

#### 获取或创建Tick数据块 getTickBlock
获取实时Tick数据映射表 `_rt_ticks_blocks` 中对应 ct 的数据块的引用。
- 如果不存在该数据块，创建 `{_base_dir}rt/ticks/{ct->getExchg()}/{ct->getCode()}.dmb` 并关系起来
- 该数据块的 `_date` 要与参数 curDate 一致（非新关联且不一致的，要用 0 覆盖这个数据块的数据）

#### 获取或创建逐笔成交数据块 getTransBlock
获取实时逐笔成交数据映射表 `_rt_trans_blocks` 中对应 ct 的数据块的引用。
- 如果不存在该数据块，创建 `{_base_dir}rt/trans/{ct->getExchg()}/{ct->getCode()}.dmb` 并关系起来
- 该数据块的 `_date` 要与参数 curDate 一致（非新关联且不一致的，要用 0 覆盖这个数据块的数据）

#### 获取或创建委托详情数据块 getOrdDtlBlock
获取实时委托数据映射表 `_rt_orddtl_blocks` 中对应 ct 的数据块的引用。
- 如果不存在该数据块，创建 `{_base_dir}rt/orders/{ct->getExchg()}/{ct->getCode()}.dmb` 并关系起来
- 该数据块的 `_date` 要与参数 curDate 一致（非新关联且不一致的，要用 0 覆盖这个数据块的数据）

#### 获取或创建委托队列数据块 getOrdQueBlock
获取实时委托队列数据映射表 `_rt_ordque_blocks` 中对应 ct 的数据块的引用。
- 如果不存在该数据块，创建 `{_base_dir}rt/queue/{ct->getExchg()}/{ct->getCode()}.dmb` 并关系起来
- 该数据块的 `_date` 要与参数 curDate 一致（非新关联且不一致的，要用 0 覆盖这个数据块的数据）

#### 调整实时数据块大小（扩容）resizeRTBlock
将实时数据块 mfPtr 的容量增加到 nCount
```cpp
template<typename HeaderType, typename T>
void* WtDataWriter::resizeRTBlock(BoostMFPtr& mfPtr, uint32_t nCount)
```

# 回测数据读取器 WtBtDtReader.h/cpp
```cpp
class WtBtDtReader : public IBtDtReader
```
参考 [Includes/note.ipynb/数据管理接口层/回测数据读取 IBtDtReader.h/回测数据读取接口类 IBtDtReader](../Includes/note.ipynb)

## 成员
- `std::string _base_dir`：数据存储基础目录路径

## 方法（IBtDtReader 接口实现）

### 初始化数据读取器 init

### 读取原始K线数据 read_raw_bars
读取 `{_base_dir}his/{period: K线周期}/{exchg}/{code}.dsb` 数据到 buffer
```cpp
bool WtBtDtReader::read_raw_bars(const char* exchg, const char* code, WTSKlinePeriod period, std::string& buffer)
```

### 读取原始Tick数据 read_raw_ticks
读取 `{_base_dir}his/ticks/{exchg}/{uDate}/{code}.dsb` 数据到 buffer
```cpp
bool WtBtDtReader::read_raw_ticks(const char* exchg, const char* code, uint32_t uDate, std::string& buffer)
```

### 读取原始委托明细数据 read_raw_order_details
读取 `{_base_dir}his/orders/{exchg}/{uDate}/{code}.dsb` 数据到 buffer
```cpp
bool WtBtDtReader::read_raw_order_details(const char* exchg, const char* code, uint32_t uDate, std::string& buffer)
```

### 读取原始委托队列数据 read_raw_order_queues
读取 `{_base_dir}his/queue/{exchg}/{uDate}/{code}.dsb` 数据到 buffer
```cpp
bool WtBtDtReader::read_raw_order_queues(const char* exchg, const char* code, uint32_t uDate, std::string& buffer)
```

### 读取原始逐笔成交数据 read_raw_transactions
读取 `{_base_dir}his/trans/{exchg}/{uDate}/{code}.dsb` 数据到 buffer
```cpp
bool WtBtDtReader::read_raw_transactions(const char* exchg, const char* code, uint32_t uDate, std::string& buffer)
```

# 随机数据读取器 WtRdmDtReader.h/cpp
基本类似于 ***数据读取器 WtDataReader.h/cpp***

```cpp
class WtRdmDtReader : public IRdmDtReader
```
参考 [Includes/note.ipynb/数据管理接口层/随机顺序数据读取 IRdmDtReader.h/随机顺序数据读取接口类 IRdmDtReader](../Includes/note.ipynb)

## 成员
-   **核心配置与管理器指针**
    -   `std::string _base_dir`：数据存储的根目录路径。
    -   `IBaseDataMgr* _base_data_mgr`：基础数据管理器指针，用于获取合约、交易日等静态基础信息。
    -   `IHotMgr* _hot_mgr`：主力合约管理器指针，用于解析 `HOT` (主力) 或 `2ND` (次主力) 等连续合约代码。
    -   `StdThreadPtr _thrd_check`：一个后台检查线程的智能指针。
    -   `bool _stopped`：线程终止标志，用于通知 `_thrd_check` 线程安全退出。

-   **实时数据块缓存**
    -   `RTKBlockFilesMap _rt_min1_map`：实时1分钟K线数据块映射表。
    -   `RTKBlockFilesMap _rt_min5_map`：实时5分钟K线数据块映射表。
        -   typedef std::unordered_map<std::string, `RTKlineBlockPair`> `RTKBlockFilesMap`; 包含:
            -   `StdUniqueMutex* _mtx`：互斥锁指针，用于保护该数据块的线程安全访问。
            -   `RTKlineBlock* _block`：指向内存映射文件中的实时K线数据块。
            -   `BoostMFPtr _file`：内存映射文件智能指针。
            -   `uint64_t _last_cap`：记录数据块最后一次的容量。
            -   `uint64_t _last_time`：记录最后访问时间戳，用于缓存清理。
    -   `TBlockFilesMap _rt_tick_map`：实时Tick数据块映射表。
        -   typedef std::unordered_map<std::string, `TickBlockPair`> `TBlockFilesMap`; 包含:
            -   `StdUniqueMutex* _mtx`：互斥锁指针。
            -   `RTTickBlock* _block`：指向实时Tick数据块。
            -   `BoostMFPtr _file`：内存映射文件智能指针。
            -   `uint64_t _last_cap`：最后容量。
            -   `uint64_t _last_time`：最后访问时间。
    -   `TransBlockFilesMap _rt_trans_map`：实时逐笔成交数据块映射表。
        -   typedef std::unordered_map<std::string, `TransBlockPair`> `TransBlockFilesMap`; 包含 (结构与 `TickBlockPair` 类似)。
    -   `OrdDtlBlockFilesMap _rt_orddtl_map`：实时逐笔委托数据块映射表。
        -   typedef std::unordered_map<std::string, `OrdDtlBlockPair`> `OrdDtlBlockFilesMap`; 包含 (结构与 `TickBlockPair` 类似)。
    -   `OrdQueBlockFilesMap _rt_ordque_map`：实时委托队列数据块映射表。
        -   typedef std::unordered_map<std::string, `OrdQueBlockPair`> `OrdQueBlockFilesMap`; 包含 (结构与 `TickBlockPair` 类似)。

-   **历史数据块缓存**
    -   `HisTickBlockMap _his_tick_map`：历史Tick数据块映射表。
        -   typedef std::unordered_map<std::string, `HisTBlockPair`> `HisTickBlockMap`; 包含:
            -   `HisTickBlock* _block`：指向已解析的历史Tick数据块内存。
            -   `uint64_t _date`：该数据块对应的交易日期。
            -   `std::string _buffer`：存储从文件读取并解压后的原始二进制数据。
    -   `HisOrdDtlBlockMap _his_orddtl_map`：历史逐笔委托数据块映射表。
        -   typedef std::unordered_map<std::string, `HisOrdDtlBlockPair`> `HisOrdDtlBlockMap`; (内部结构与 `HisTBlockPair` 类似)。
    -   `HisOrdQueBlockMap _his_ordque_map`：历史委托队列数据块映射表。
        -   typedef std::unordered_map<std::string, `HisOrdQueBlockPair`> `HisOrdQueBlockMap`; (内部结构与 `HisTBlockPair` 类似)。
    -   `HisTransBlockMap _his_trans_map`：历史逐笔成交数据块映射表。
        -   typedef std::unordered_map<std::string, `HisTransBlockPair`> `HisTransBlockMap`; (内部结构与 `HisTBlockPair` 类似)。

-   **K线与复权数据缓存**
    -   `BarsCache _bars_cache`：K线数据缓存。这是 `WtRdmDtReader` 最核心的缓存之一，用于存储从文件加载并可能经过复权、主力合约拼接等复杂处理后的完整K线序列。
        -   typedef std::unordered_map<std::string, `BarsList`> `BarsCache`;
            -   `std::string _exchg`：交易所代码。
            -   `std::string _code`：合约代码 (可能是连续合约代码, 如 `SHFE.rb.HOT`)。
            -   `WTSKlinePeriod _period`：K线周期。
            -   `std::string _raw_code`：当前日期对应的实际分月合约代码 (如 `rb2410`)。
            -   `double _factor`：后复权因子。
            -   `std::vector<WTSBarStruct> _bars`：存储历史K线数据。
            -   `std::vector<WTSBarStruct> _rt_bars`：当进行后复权时，用于单独缓存并处理实时K线数据。
    -   `AdjFactorMap _adj_factors`：复权因子映射表。
        -   **作用**: 专门用于缓存股票的除权除息数据，是进行K线复权计算的基础。
        -   typedef std::unordered_map<std::string, `AdjFactorList`> `AdjFactorMap`;
            -   `typedef std::vector<AdjFactor> AdjFactorList`;
                -   `uint32_t _date`：复权因子生效的日期。
                -   `double _factor`：复权因子值。

## 方法

### IRdmDtReader 接口实现

#### 初始化 init

#### 清理缓存 clearCache

#### 按时间范围读取Tick数据切片 readTickSliceByRange
流程：
- 构建Tick切片 slice: WTSTickSlice*
- **处理当天之前的所有日期**
  - 如果是期货连续合约（HOT或2ND）：构建 hotCode
  - 构建缓存键 key = stdCode.当天日期
  - 如果历史Tick数据块映射表中**不存在** `_his_tick_map[key]` 数据块
    - 如果是期货连续合约：
      - 从 `{_base_dir}/his/ticks/{交易所}/{当天日期}/{hotCode}.dsb` 中读取并关联
    - 如果不是期货连续合约
      - 从 `{_base_dir}/his/ticks/{交易所}/{当天日期}/{合约代码}.dsb` 中读取并关联
  - 如果读取到了
    - 根据当前处理的日期构造一个 eTick: WTSTickStruct
    - 查找 eTick 在数据块 `_his_tick_map[key]` 中的对应位置 eIdx
      - 如果当前处理日期不是开始日期：将 eIdx 之前的所有Tick数据添加到 slice
      - 否则：查找 sTick 在数据块 `_his_tick_map[key]` 中的对应位置 sIdx，将 sIdx 和 eIdx 之间的Tick数据添加到 slice
- **处理当天**
  - 获取 `_rt_tick_map` 中对应的数据块
  - 根据当前处理的日期构造一个 eTick: WTSTickStruct
  - 查找 eTick 在该数据块中的对应位置 eIdx
    - 如果当前处理日期不是开始日期：将 eIdx 之前的所有Tick数据添加到 slice
    - 否则：查找 sTick 在该数据块中的对应位置 sIdx，将 sIdx 和 eIdx 之间的Tick数据添加到 slice
- 返回 slice
```cpp
/*!
 * \brief 按时间范围读取Tick数据切片
 * \param stdCode 标准合约代码
 * \param stime 开始时间（微秒时间戳）
 * \param etime 结束时间（微秒时间戳，默认为0）
 * \return Tick数据切片指针，失败返回NULL
 */
WTSTickSlice* WtRdmDtReader::readTickSliceByRange(const char* stdCode, uint64_t stime, uint64_t etime /* = 0 */)
```

#### 按时间范围读取委托明细数据切片 readOrdDtlSliceByRange

#### 按时间范围读取委托队列数据切片 readOrdQueSliceByRange

#### 按时间范围读取逐笔成交数据切片 readTransSliceByRange

#### 按时间范围读取K线数据切片 readKlineSliceByRange

#### 按日期读取Tick数据切片 readTickSliceByDate

#### 按数量读取Tick数据切片 readTickSliceByCount

#### 按数量读取K线数据切片 readKlineSliceByCount

#### 获取个股指定日期的复权因子 getAdjFactorByDate

### 实时数据块获取 

#### 获取实时K线数据块 getRTKilneBlock

#### 获取实时Tick数据块 getRTTickBlock

#### 获取实时委托队列数据块 getRTOrdQueBlock

#### 获取实时逐笔委托数据块 getRTOrdDtlBlock

#### 获取实时逐笔成交数据块 getRTTransBlock

### 数据缓存与索引

#### 从文件缓存历史K线数据 cacheHisBarsFromFile

#### 从缓存按时间范围读取K线数据 readBarsFromCacheByRange

#### 从缓存按时间范围索引K线数据 indexBarFromCacheByRange

#### 从缓存按数量索引K线数据 indexBarFromCacheByCount

### 复权因子处理

#### 从文件加载股票复权因子 loadStkAdjFactorsFromFile

#### 获取复权因子列表 getAdjFactors